<div style="background: linear-gradient(135deg, #0a1a0a 0%, #1a3a1a 50%, #0d2b0d 100%); padding: 45px 20px; border-radius: 12px; text-align: center; font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; box-shadow: 0 12px 24px rgba(0,0,0,0.6); border: 1px solid rgba(0,230,118,0.15); position: relative; overflow: hidden;">
    <div style="position: absolute; top: -30px; left: -20px; width: 120px; height: 120px; background: radial-gradient(circle, rgba(0,230,118,0.08) 0%, transparent 70%); border-radius: 50%;"></div>
    <div style="position: absolute; bottom: -30px; right: -20px; width: 150px; height: 150px; background: radial-gradient(circle, rgba(29,233,182,0.07) 0%, transparent 70%); border-radius: 50%;"></div>
    <div style="font-size: 36px; margin-bottom: 10px;">🦜</div>
    <h1 style="color: #ffffff; font-size: 38px; margin: 0 0 12px 0; font-weight: 800; letter-spacing: 1.5px; text-shadow: 0px 4px 10px rgba(0,0,0,0.8);">BirdCLEF+ 2026</h1>
    <div style="height: 4px; width: 80px; background: linear-gradient(to right, #00e676, #1de9b6); margin: 0 auto 15px auto; border-radius: 2px; box-shadow: 0 0 10px rgba(0,230,118,0.5);"></div>
    <p style="color: #a5d6a7; font-size: 16px; margin: 0; letter-spacing: 3px; text-transform: uppercase; font-weight: 500;">Acoustic Species Identification · Pantanal, Brazil</p>
</div>

# 🦜 BirdCLEF+ 2026 — Perch v2 + ProtoSSM v5 · Multi-Seed Ensemble
 
Multi-label wildlife species identification from passive acoustic recordings in Brazil's Pantanal wetlands. 234 species across birds, amphibians, mammals, reptiles, and insects.
 
**Public Score:** 0.927 → 0.933+ (multi-seed ensemble)
**Runtime:** ~18 minutes on CPU
**Author:** Imaad Mahmood
 
---
 
## Approach
 
Maximum ensemble pipeline built on top of Google Perch v2:
 
**1. Google Perch v2 (frozen)**
Audio foundation model pretrained on 14,795 species. Combined with Bayesian site × hour-of-day priors, smooth temporal averaging, and genus-level proxy mapping for 31 unmapped species.
 
**2. ProtoSSM v5 — Multi-Seed Ensemble**
Mamba-style bidirectional Selective State Space Model (d_model=320, 4 BiSSM layers, cross-attention, 8 heads). Trained 3 times with different random seeds (42, 123, 777). Each model gets different mixup augmentation trajectories. Final proto score = average of all 3 TTA predictions. Validation split (15% of files) ensures early stopping is meaningful.
 
**3. MLP + LGBM Probe Ensemble**
Per-class MLP (256,128) and LightGBM probes trained on 128-dim PCA-compressed Perch embeddings with 15 sequential + interaction features. MLP uses positive-class oversampling. Blended 70% MLP / 30% LGBM. 58 classes modeled.
 
**4. ResidualSSM**
Lightweight second-pass BiSSM that learns to correct systematic first-pass errors. Trained on residuals (ground truth − sigmoid(first pass)).
 
**5. Post-processing pipeline**
Per-taxon temperature → File-level confidence scaling → Rank-aware scaling → Adaptive delta shift smoothing → Isotonic calibration (per-class) → Per-class threshold sharpening.
 
---
 
## Results
 
| Version | Model | Score |
|---|---|---|
| v1 | EfficientNet-B1 baseline | 0.798 |
| v2 | B1 + TTA + Gaussian smoothing | 0.826 |
| v3 | Perch v2 + Bayesian priors + LogReg probes | 0.912 |
| v4 | ProtoSSM v5 + MLP/LGBM probes + ResidualSSM | 0.927 |
| v5 | + Multi-seed ensemble + proper val split + isotonic cal | **0.933+** |
 
---
 
## Key Improvements in v5
 
| Fix | Why | Expected gain |
|---|---|---|
| Multi-seed ProtoSSM (3 seeds) | Different mixup → model diversity → averaging reduces variance | +0.004 |
| Proper validation split (15% files) | ProtoSSM early stopping was blind (val_auc=0.0 before) | +0.002 |
| Isotonic calibration | Corrects systematic over/underconfidence per species | +0.002 |
 
---
 
## Required Inputs
 
| Type | Name |
|---|---|
| Competition | birdclef-2026 |
| Model | google/bird-vocalization-classifier (perch_v2_cpu/1) |
| Notebook | ashok205/tf-wheels |
| Dataset | jaejohn/perch-meta |
 
---
 
## Pipeline
 
```
Google Perch v2 (frozen, 14,795 species)
    ├── Raw logits → Bayesian site × hour prior fusion
    └── 1536-dim embeddings → PCA (128 dims) → MLP+LGBM probes (58 classes)
 
ProtoSSM v5 × 3 seeds (d_model=320, 4 BiSSM layers)
    ├── Each seed: different mixup augmentation
    ├── Proper 85/15 train/val split
    ├── Focal loss + SWA + 5-shift TTA
    └── Average across seeds
 
Ensemble = 0.50 × ProtoSSM_avg + 0.50 × MLP/LGBM
                ↓
         ResidualSSM (second-pass error correction)
                ↓
    Per-taxon temperature → File-level scaling →
    Rank-aware scaling → Delta shift smooth →
    Isotonic calibration → Threshold sharpening
                ↓
              submission.csv
```
 
---
 
## References
 
- [Pantanal Distill BirdCLEF2026 ONNX](https://www.kaggle.com/code/dingjiarun/pantanal-distill-birdclef2026-onnx?scriptVersionId=307575470) by dingjiarun — core ProtoSSM architecture and post-processing pipeline
- [Perch v2 Starter: Train + Infer](https://www.kaggle.com/code/jaejohn/perch-v2-starter-train-infer) by jaejohn — Perch embedding cache
- [tf-wheels](https://www.kaggle.com/code/ashok205/tf-wheels) by ashok205 — TF 2.20 wheels
- [Google Bird Vocalization Classifier (Perch)](https://kaggle.com/models/google/bird-vocalization-classifier)
 

In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — ProtoSSM + ONNX Perch → submission_protossm.csv
# Author: Imaad Mahmood
# ═══════════════════════════════════════════════════════════════
import os, re, gc, time, warnings, subprocess, sys
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

INPUT_ROOT = Path("/kaggle/input")
BASE       = Path("/kaggle/input/competitions/birdclef-2026")
WORK_DIR   = Path("/kaggle/working/cache")
WORK_DIR.mkdir(parents=True, exist_ok=True)

ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026"
                "/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")

def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(pattern)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)

import onnxruntime as ort
import tensorflow as tf
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
import torch
import torch.nn as nn
import torch.nn.functional as F

tf.experimental.numpy.experimental_enable_numpy_behavior()

# ── ANSI ─────────────────────────────────────────────────────────────────────
G='\033[92m'; C='\033[96m'; Y='\033[93m'; W='\033[97m'
B='\033[94m'; M='\033[95m'; RS='\033[0m'; BD='\033[1m'
def banner(t,color=C): w=65; print(f'\n{color}{BD}{"═"*w}{RS}\n{color}{BD}  {t}{RS}\n{color}{BD}{"═"*w}{RS}')
def step(t,c=B): print(f'{c}{BD}▶  {t}{RS}')
def ok(t): print(f'{G}{BD}✅  {t}{RS}')
def info(k,v,c=W): print(f'  {Y}{k:<28}{RS}{c}{v}{RS}')

_WALL = time.time()
banner('Cell 1 — ProtoSSM + ONNX Perch', C)

SR=32000; WINDOW_SEC=5; WINDOW_SAMPLES=SR*WINDOW_SEC; FILE_SAMPLES=60*SR; N_WINDOWS=12

CFG = {
    "batch_files": 16,
    "proto_epochs": 40,
    "proto_patience": 8,
    "res_epochs": 20,
    "res_patience": 6,
    "tta_shifts":  [0, 1, -1, 2, -2],
    "tta_weights": [0.4, 0.2, 0.2, 0.1, 0.1],
}

# ── Data ──────────────────────────────────────────────────────────────────────
taxonomy          = pd.read_csv(BASE / "taxonomy.csv")
sample_sub        = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")
PRIMARY_LABELS    = sample_sub.columns[1:].tolist()
N_CLASSES         = len(PRIMARY_LABELS)
label_to_idx      = {c: i for i, c in enumerate(PRIMARY_LABELS)}
FNAME_RE          = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(str(name))
    if not m: return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t: out.add(t)
    return sorted(out)

sc = (soundscape_labels.groupby(["filename", "start", "end"])["primary_label"]
      .apply(union_labels).reset_index(name="label_list"))
sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"]  = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)
sc = pd.concat([sc, sc["filename"].apply(parse_fname).apply(pd.Series)], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in label_to_idx: Y_SC[i, label_to_idx[lbl]] = 1

wpf        = sc.groupby("filename").size()
full_files = sorted(wpf[wpf == N_WINDOWS].index.tolist())
sc["fully_labeled"] = sc["filename"].isin(full_files)
full_rows = sc[sc["fully_labeled"]].sort_values(["filename", "end_sec"]).reset_index(drop=False)
Y_FULL    = Y_SC[full_rows["index"].to_numpy()]

info("Classes",        str(N_CLASSES),                         G)
info("Full files",     str(len(full_files)),                   G)
info("Active classes", str(int((Y_FULL.sum(0) > 0).sum())),    G)

# ── Perch / ONNX setup ───────────────────────────────────────────────────────
MODEL_DIR       = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
# Prefer no-DFT variant (scores higher), fall back to standard
ONNX_PERCH_PATH = next(INPUT_ROOT.glob("**/perch_v2_no_dft*.onnx"),
                   next(INPUT_ROOT.glob("**/perch_v2*.onnx"), Path("")))
USE_ONNX        = ONNX_PERCH_PATH.exists()

if USE_ONNX:
    so = ort.SessionOptions(); so.intra_op_num_threads = 4
    ONNX_SESSION    = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=so,
                                            providers=["CPUExecutionProvider"])
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    info("Perch backend", "ONNX ✅", G)
else:
    birdclassifier = tf.saved_model.load(str(MODEL_DIR))
    infer_fn       = birdclassifier.signatures["serving_default"]
    info("Perch backend", "TF SavedModel", Y)

bc_labels = (pd.read_csv(MODEL_DIR / "assets" / "labels.csv").reset_index()
             .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}))
NO_LABEL  = len(bc_labels)
mapping   = taxonomy.merge(bc_labels.rename(columns={"scientific_name": "scientific_name"}),
                            on="scientific_name", how="left")
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc    = mapping.set_index("primary_label")["bc_index"]
BC_INDICES     = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK    = BC_INDICES != NO_LABEL
MAPPED_POS     = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX  = BC_INDICES[MAPPED_MASK].astype(np.int32)
UNMAPPED_POS   = np.where(~MAPPED_MASK)[0].astype(np.int32)
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}

proxy_map = {}
for _, row in taxonomy[taxonomy["primary_label"].isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].iterrows():
    genus = str(row["scientific_name"]).split()[0]
    hits  = bc_labels[bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)]
    if len(hits) and CLASS_NAME_MAP.get(row["primary_label"]) in {"Amphibia","Insecta","Aves"}:
        proxy_map[label_to_idx[row["primary_label"]]] = hits["bc_index"].astype(int).tolist()

TAXON_TEMPS = {"Aves": 0.90, "Amphibia": 1.10, "Insecta": 1.15, "Mammalia": 1.00, "Reptilia": 1.00}
CLASS_TEMPERATURES = np.ones(N_CLASSES, dtype=np.float32)
for i, lbl in enumerate(PRIMARY_LABELS):
    CLASS_TEMPERATURES[i] = TAXON_TEMPS.get(CLASS_NAME_MAP.get(lbl, ""), 1.0)

info("Mapped species", f"{MAPPED_MASK.sum()}/{N_CLASSES}", G)
info("Proxy targets",  str(len(proxy_map)),                G)

# ── Perch inference ───────────────────────────────────────────────────────────
def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES: y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    return y[:FILE_SAMPLES].astype(np.float32)

def run_perch(paths, batch_files=16, verbose=True):
    paths  = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS
    row_ids=np.empty(n_rows,dtype=object); filenames=np.empty(n_rows,dtype=object)
    sites=np.empty(n_rows,dtype=object);   hours=np.zeros(n_rows,dtype=np.int16)
    scores=np.zeros((n_rows,N_CLASSES),dtype=np.float32)
    embs=np.zeros((n_rows,1536),dtype=np.float32)
    wr=0; itr=tqdm(range(0,len(paths),batch_files),desc=f"{M}Perch{RS}") if verbose else range(0,len(paths),batch_files)
    for start in itr:
        batch_paths=paths[start:start+batch_files]
        batch_audio=[read_60s(p) for p in batch_paths]
        x=np.empty((len(batch_paths)*N_WINDOWS,WINDOW_SAMPLES),dtype=np.float32); br=wr
        for bi,path in enumerate(batch_paths):
            meta=parse_fname(path.name); stem=path.stem
            x[bi*N_WINDOWS:(bi+1)*N_WINDOWS]=batch_audio[bi].reshape(N_WINDOWS,WINDOW_SAMPLES)
            row_ids[wr:wr+N_WINDOWS]=[f"{stem}_{t}" for t in range(5,65,5)]
            filenames[wr:wr+N_WINDOWS]=path.name; sites[wr:wr+N_WINDOWS]=meta["site"]
            hours[wr:wr+N_WINDOWS]=meta["hour_utc"]; wr+=N_WINDOWS
        if USE_ONNX:
            outs=ONNX_SESSION.run(None,{ONNX_INPUT_NAME:x})
            logits=outs[ONNX_OUT_MAP.get("label",0)].astype(np.float32)
            emb=outs[ONNX_OUT_MAP.get("embedding",1)].astype(np.float32)
        else:
            out=infer_fn(inputs=tf.convert_to_tensor(x))
            logits=out["label"].numpy().astype(np.float32)
            emb=out["embedding"].numpy().astype(np.float32)
        scores[br:wr,MAPPED_POS]=logits[:,MAPPED_BC_IDX]; embs[br:wr]=emb
        for pos_idx,bc_idxs in proxy_map.items():
            scores[br:wr,pos_idx]=logits[:,np.asarray(bc_idxs,dtype=np.int32)].max(axis=1)
        del x,logits,emb,batch_audio; gc.collect()
    return pd.DataFrame({"row_id":row_ids,"filename":filenames,"site":sites,"hour_utc":hours}),scores,embs

# ── Cache ──────────────────────────────────────────────────────────────────────
SCORE_KEYS=["scores","scores_full_raw","sc","logits","perch_scores","arr_0"]
EMB_KEYS  =["embs","emb_full","emb","embeddings","features","arr_1"]

def pick_array(arr,candidates,cols):
    for k in candidates:
        if k in arr.files: return arr[k],k
    for k in arr.files:
        v=arr[k]
        if v.ndim==2 and v.shape[1]==cols: return v,k
    raise KeyError(f"No array for cols={cols}; keys={list(arr.files)}")

def find_ext_cache():
    for d in INPUT_ROOT.glob("**"):
        if not d.is_dir(): continue
        for mp,np_ in [("perch_meta.parquet","perch_arrays.npz"),
                       ("full_perch_meta.parquet","full_perch_arrays.npz")]:
            if (d/mp).exists() and (d/np_).exists(): return d/mp,d/np_
    return None,None

CACHE_META_L=WORK_DIR/"perch_meta.parquet"; CACHE_NPZ_L=WORK_DIR/"perch_arrays.npz"
ext_meta,ext_npz=find_ext_cache()
if ext_meta: CACHE_META,CACHE_NPZ=ext_meta,ext_npz; step(f"External cache: {ext_meta.parent}")
elif CACHE_META_L.exists() and CACHE_NPZ_L.exists(): CACHE_META,CACHE_NPZ=CACHE_META_L,CACHE_NPZ_L; step("Local cache")
else:
    step("Building cache from scratch...")
    train_paths=[BASE/"train_soundscapes"/fn for fn in full_files if (BASE/"train_soundscapes"/fn).exists()]
    meta_b,sc_b,emb_b=run_perch(train_paths,verbose=True)
    meta_b.to_parquet(CACHE_META_L)
    np.savez(CACHE_NPZ_L,scores=sc_b.astype(np.float32),embs=emb_b.astype(np.float32),
             primary_labels=np.array(PRIMARY_LABELS))
    CACHE_META,CACHE_NPZ=CACHE_META_L,CACHE_NPZ_L; ok("Cache saved")

meta_tr=pd.read_parquet(CACHE_META); arr=np.load(CACHE_NPZ)
sc_tr,_=pick_array(arr,SCORE_KEYS,N_CLASSES); emb_tr,_=pick_array(arr,EMB_KEYS,1536)
sc_tr=sc_tr.astype(np.float32); emb_tr=emb_tr.astype(np.float32)
if "row_id" not in meta_tr.columns:
    end_sec=(meta_tr["end_sec"].astype(int) if "end_sec" in meta_tr.columns
             else np.tile(np.arange(5,65,5),len(meta_tr)//N_WINDOWS))
    meta_tr["row_id"]=meta_tr["filename"].str.replace(".ogg","",regex=False)+"_"+end_sec.astype(str)
row_id_to_index=full_rows.set_index("row_id")["index"]
Y_FULL_aligned=Y_SC[row_id_to_index.loc[meta_tr["row_id"]].to_numpy()].astype(np.uint8)
info("sc_tr",  str(sc_tr.shape),  G)
info("emb_tr", str(emb_tr.shape), G)

# ── Post-processing helpers ───────────────────────────────────────────────────
def sigmoid(x): return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))

def build_prior_tables(meta,Y,smooth_site=20,smooth_hour=30):
    df=meta.copy().reset_index(drop=True); df["site"]=df["site"].astype(str); df["hour_utc"]=df["hour_utc"].astype(int)
    gp=np.log((Y.mean(0)+1e-4)/(1-Y.mean(0)+1e-4))
    sp,hp={},{}
    for site,idx in df.groupby("site").indices.items():
        y=Y[list(idx)]; p=(y.sum(0)+smooth_site*Y.mean(0))/(len(idx)+smooth_site)
        sp[site]=np.log((p+1e-4)/(1-p+1e-4))
    for hour,idx in df.groupby("hour_utc").indices.items():
        y=Y[list(idx)]; p=(y.sum(0)+smooth_hour*Y.mean(0))/(len(idx)+smooth_hour)
        hp[int(hour)]=np.log((p+1e-4)/(1-p+1e-4))
    return {"global":gp.astype(np.float32),"site":sp,"hour":hp}

def apply_prior(scores,sites,hours,tables,lp=0.4):
    out=scores.copy()
    for i,(s,h) in enumerate(zip(sites,hours)):
        prior=tables["global"].copy()
        if str(s) in tables["site"]: prior=0.5*prior+0.5*tables["site"][str(s)]
        if int(h) in tables["hour"]: prior=0.5*prior+0.5*tables["hour"][int(h)]
        out[i]+=lp*prior
    return out

def file_confidence_scale(probs,n_windows=12,alpha=0.15):
    v=probs.reshape(-1,n_windows,probs.shape[1])
    top2=np.sort(v,axis=1)[:,-2:].mean(axis=1,keepdims=True)
    return (v*((1-alpha)+alpha*top2)).reshape(probs.shape)

def rank_aware_scaling(probs,n_windows=12,power=0.4):
    v=probs.reshape(-1,n_windows,probs.shape[1])
    return (v*np.power(v.max(axis=1,keepdims=True),power)).reshape(probs.shape)

def adaptive_delta_smooth(probs,n_windows=12,base_alpha=0.20):
    out=probs.copy(); v=probs.reshape(-1,n_windows,probs.shape[1]); o=out.reshape(-1,n_windows,probs.shape[1])
    for t in range(n_windows):
        conf=v[:,t].max(axis=-1,keepdims=True); alpha=base_alpha*(1-conf)
        neigh=((v[:,t]+v[:,t+1])/2 if t==0 else (v[:,t-1]+v[:,t])/2 if t==n_windows-1
               else (v[:,t-1]+v[:,t+1])/2)
        o[:,t]=(1-alpha)*v[:,t]+alpha*neigh
    return out

def calibrate_and_optimize_thresholds(oof_probs,Y_FULL,threshold_grid=None,n_windows=12):
    if threshold_grid is None: threshold_grid=[0.25,0.30,0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]
    n_cls=oof_probs.shape[1]; thresholds=np.full(n_cls,0.5,dtype=np.float32)
    file_oof=oof_probs.reshape(-1,n_windows,n_cls).max(axis=1)
    file_y=Y_FULL.reshape(-1,n_windows,n_cls).max(axis=1)
    for c in range(n_cls):
        y_true,y_prob=file_y[:,c],file_oof[:,c]
        if y_true.sum()<3: continue
        try: ir=IsotonicRegression(out_of_bounds="clip"); ir.fit(y_prob,y_true); y_cal=ir.transform(y_prob)
        except: y_cal=y_prob
        best_f1,best_t=0.0,0.5
        for t in threshold_grid:
            pred=(y_cal>=t).astype(int)
            tp=((pred==1)&(y_true==1)).sum(); fp=((pred==1)&(y_true==0)).sum(); fn=((pred==0)&(y_true==1)).sum()
            prec=tp/(tp+fp+1e-8); rec=tp/(tp+fn+1e-8); f1=2*prec*rec/(prec+rec+1e-8)
            if f1>best_f1: best_f1,best_t=f1,t
        thresholds[c]=best_t
    return thresholds

def apply_per_class_thresholds(scores,thresholds):
    scaled=scores.copy()
    for c in range(scores.shape[1]):
        t=thresholds[c]; above=scores[:,c]>t
        scaled[above,c]=0.5+0.5*(scores[above,c]-t)/(1-t+1e-8)
        scaled[~above,c]=0.5*scores[~above,c]/(t+1e-8)
    return np.clip(scaled,0,1)

# ── MLP probes ────────────────────────────────────────────────────────────────
def build_class_freq_weights(Y,cap=10.0):
    pos=Y.sum(0).astype(np.float32)+1.0; freq=pos/Y.shape[0]
    w=np.clip(1.0/(freq**0.5),1.0,cap); return (w/w.mean()).astype(np.float32)

def build_seq_features(scores_col,n_windows=12):
    x=scores_col.reshape(-1,n_windows)
    prev=np.concatenate([x[:,:1],x[:,:-1]],1); nxt=np.concatenate([x[:,1:],x[:,-1:]],1)
    return prev.reshape(-1),nxt.reshape(-1),np.repeat(x.mean(1),n_windows),np.repeat(x.max(1),n_windows),np.repeat(x.std(1),n_windows)

def train_mlp_probes(emb,scores_raw,Y,min_pos=5,pca_dim=128,alpha_blend=0.4):
    scaler=StandardScaler(); emb_s=scaler.fit_transform(emb)
    pca=PCA(n_components=min(pca_dim,emb_s.shape[1]-1)); Z=pca.fit_transform(emb_s).astype(np.float32)
    info("Embedding→PCA",f"{emb.shape}→{Z.shape} ({pca.explained_variance_ratio_.sum():.2%})",G)
    cw=build_class_freq_weights(Y,cap=10.0); active=np.where(Y.sum(0)>=min_pos)[0]; probe_models={}; MAX_ROWS=3000
    for ci in tqdm(active,desc=f"{B}MLP probes{RS}"):
        y=Y[:,ci]
        if y.sum()==0 or y.sum()==len(y): continue
        prev,nxt,mean,mx,std=build_seq_features(scores_raw[:,ci])
        X=np.hstack([Z,scores_raw[:,ci:ci+1],prev[:,None],nxt[:,None],mean[:,None],mx[:,None],std[:,None]])
        n_pos,n_neg=int(y.sum()),len(y)-int(y.sum()); pos_idx=np.where(y==1)[0]
        repeat=max(1,min(int(round(float(cw[ci])*n_neg/max(n_pos,1))),8))
        if n_pos*repeat+len(y)>MAX_ROWS: repeat=max(1,(MAX_ROWS-len(y))//max(n_pos,1))
        X_bal=np.vstack([X,np.tile(X[pos_idx],(repeat,1))])
        y_bal=np.concatenate([y,np.ones(n_pos*repeat,dtype=y.dtype)])
        clf=MLPClassifier(hidden_layer_sizes=(128,64),activation="relu",max_iter=200,
                          early_stopping=True,validation_fraction=0.15,n_iter_no_change=10,
                          random_state=42,learning_rate_init=5e-4,alpha=0.005)
        clf.fit(X_bal,y_bal); probe_models[ci]=clf
    info("MLP probes",f"{len(probe_models)}/{N_CLASSES}",G)
    return probe_models,scaler,pca,alpha_blend

class VectorizedMLPProbes(nn.Module):
    def __init__(self,probe_models):
        super().__init__(); self.valid_classes=sorted(probe_models.keys())
        if not self.valid_classes: self.n_layers=0; self.weights=nn.ParameterList(); self.biases=nn.ParameterList(); return
        sample=probe_models[self.valid_classes[0]]; self.n_layers=len(sample.coefs_)
        self.weights=nn.ParameterList(); self.biases=nn.ParameterList()
        for li in range(self.n_layers):
            W=np.stack([probe_models[c].coefs_[li] for c in self.valid_classes],0)
            b=np.stack([probe_models[c].intercepts_[li] for c in self.valid_classes],0)
            self.weights.append(nn.Parameter(torch.tensor(W,dtype=torch.float32),requires_grad=False))
            self.biases.append(nn.Parameter(torch.tensor(b,dtype=torch.float32),requires_grad=False))
    def forward(self,x):
        h=x
        for i in range(self.n_layers):
            h=torch.bmm(h,self.weights[i])+self.biases[i].unsqueeze(1)
            if i<self.n_layers-1: h=torch.relu(h)
        return h.squeeze(-1)

def apply_mlp_probes_vectorized(emb_test,scores_test,probe_models,scaler,pca,alpha_blend=0.4):
    if not probe_models: return scores_test.copy()
    Z_test=pca.transform(scaler.transform(emb_test)).astype(np.float32)
    valid_classes=sorted(probe_models.keys()); V=len(valid_classes); N=len(scores_test)
    raw=scores_test[:,valid_classes].T; n_files=N//N_WINDOWS; raw_view=raw.reshape(V,n_files,N_WINDOWS)
    prev=np.concatenate([raw_view[:,:,:1],raw_view[:,:,:-1]],2).reshape(V,N)
    nxt=np.concatenate([raw_view[:,:,1:],raw_view[:,:,-1:]],2).reshape(V,N)
    mean=np.repeat(raw_view.mean(2),N_WINDOWS,1); mx=np.repeat(raw_view.max(2),N_WINDOWS,1)
    std=np.repeat(raw_view.std(2),N_WINDOWS,1)
    X_all=np.concatenate([np.broadcast_to(Z_test,(V,N,Z_test.shape[1])).astype(np.float32),
                           np.stack([raw,prev,nxt,mean,mx,std],axis=-1).astype(np.float32)],axis=-1)
    vec=VectorizedMLPProbes(probe_models).eval()
    with torch.no_grad(): preds=vec(torch.tensor(X_all)).numpy()
    result=scores_test.copy()
    result[:,valid_classes]=(1-alpha_blend)*scores_test[:,valid_classes]+alpha_blend*preds.T
    return result

# ── SSM Architecture ─────────────────────────────────────────────────────────
class SelectiveSSM(nn.Module):
    def __init__(self,d_model,d_state=16,d_conv=4):
        super().__init__(); self.d_model=d_model; self.d_state=d_state
        self.in_proj=nn.Linear(d_model,2*d_model,bias=False)
        self.conv1d=nn.Conv1d(d_model,d_model,d_conv,padding=d_conv-1,groups=d_model)
        self.dt_proj=nn.Linear(d_model,d_model,bias=True)
        A=torch.arange(1,d_state+1,dtype=torch.float32).unsqueeze(0).expand(d_model,-1)
        self.A_log=nn.Parameter(torch.log(A)); self.D=nn.Parameter(torch.ones(d_model))
        self.B_proj=nn.Linear(d_model,d_state,bias=False); self.C_proj=nn.Linear(d_model,d_state,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
    def forward(self,x):
        B_sz,T,D=x.shape; xz=self.in_proj(x); x_ssm,z=xz.chunk(2,dim=-1)
        x_conv=F.silu(self.conv1d(x_ssm.transpose(1,2))[:,:,:T].transpose(1,2))
        dt=F.softplus(self.dt_proj(x_conv)); A=-torch.exp(self.A_log)
        Bv=self.B_proj(x_conv); Cv=self.C_proj(x_conv)
        h=torch.zeros(B_sz,D,self.d_state,device=x.device,dtype=x.dtype); ys=[]
        for t in range(T):
            h=h*torch.exp(A[None]*dt[:,t,:,None])+x[:,t,:,None]*dt[:,t,:,None]*Bv[:,t,None,:]
            ys.append((h*Cv[:,t,None,:]).sum(-1))
        return torch.stack(ys,dim=1)+x*self.D[None,None,:]

class LightProtoSSM(nn.Module):
    def __init__(self,d_input=1536,d_model=128,d_state=16,n_classes=234,n_windows=12,
                 dropout=0.15,n_sites=20,meta_dim=16,use_cross_attn=True,cross_attn_heads=2):
        super().__init__(); self.n_classes=n_classes; self.use_cross_attn=use_cross_attn
        self.input_proj=nn.Sequential(nn.Linear(d_input,d_model),nn.LayerNorm(d_model),nn.GELU(),nn.Dropout(dropout))
        self.pos_enc=nn.Parameter(torch.randn(1,n_windows,d_model)*0.02)
        self.site_emb=nn.Embedding(n_sites,meta_dim); self.hour_emb=nn.Embedding(24,meta_dim)
        self.meta_proj=nn.Linear(2*meta_dim,d_model)
        self.ssm_fwd=nn.ModuleList([SelectiveSSM(d_model,d_state) for _ in range(2)])
        self.ssm_bwd=nn.ModuleList([SelectiveSSM(d_model,d_state) for _ in range(2)])
        self.ssm_merge=nn.ModuleList([nn.Linear(2*d_model,d_model) for _ in range(2)])
        self.ssm_norm=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.drop=nn.Dropout(dropout)
        if use_cross_attn:
            self.cross_attn=nn.ModuleList([nn.MultiheadAttention(d_model,cross_attn_heads,dropout=dropout,batch_first=True) for _ in range(2)])
            self.cross_norm=nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.prototypes=nn.Parameter(torch.randn(n_classes,d_model)*0.02)
        self.proto_scale=nn.Parameter(torch.ones(n_classes))
        self.proto_bias=nn.Parameter(torch.zeros(n_classes))
        self.direct_head=nn.Linear(d_model,n_classes)
        self.base_fusion=nn.Parameter(torch.zeros(n_classes))
    def forward(self,emb,perch_scores,site_ids=None,hours=None):
        B,T,_=emb.shape; h=self.input_proj(emb)+self.pos_enc[:,:T]
        if site_ids is not None and hours is not None:
            meta=torch.cat([self.site_emb(site_ids.clamp(0,self.site_emb.num_embeddings-1)),
                             self.hour_emb(hours.clamp(0,23))],dim=-1)
            h=h+self.meta_proj(meta).unsqueeze(1)
        for i in range(2):
            res=h; hf=self.ssm_fwd[i](h); hb=self.ssm_bwd[i](h.flip(1)).flip(1)
            h=self.ssm_norm[i](self.drop(self.ssm_merge[i](torch.cat([hf,hb],dim=-1)))+res)
            if self.use_cross_attn:
                res=h; h_att,_=self.cross_attn[i](h,h,h); h=self.cross_norm[i](res+self.drop(h_att))
        h_norm=F.normalize(h,dim=-1); p_norm=F.normalize(self.prototypes,dim=-1)
        proto_logits=torch.matmul(h_norm,p_norm.T)*self.proto_scale+self.proto_bias
        direct_logits=self.direct_head(h)
        alpha=torch.sigmoid(self.base_fusion)
        return 0.4*proto_logits+0.4*direct_logits+0.2*perch_scores*alpha

class ResidualSSM(nn.Module):
    def __init__(self,d_input=1536,d_scores=234,d_model=128,d_state=16,n_classes=234,
                 n_windows=12,dropout=0.1,n_sites=20,meta_dim=8):
        super().__init__()
        self.input_proj=nn.Sequential(nn.Linear(d_input+d_scores,d_model),nn.LayerNorm(d_model),nn.GELU(),nn.Dropout(dropout))
        self.site_emb=nn.Embedding(n_sites,meta_dim); self.hour_emb=nn.Embedding(24,meta_dim)
        self.meta_proj=nn.Linear(2*meta_dim,d_model)
        self.pos_enc=nn.Parameter(torch.randn(1,n_windows,d_model)*0.02)
        self.ssm_fwd=SelectiveSSM(d_model,d_state); self.ssm_bwd=SelectiveSSM(d_model,d_state)
        self.ssm_merge=nn.Linear(2*d_model,d_model); self.ssm_norm=nn.LayerNorm(d_model); self.ssm_drop=nn.Dropout(dropout)
        self.output_head=nn.Linear(d_model,n_classes)
        nn.init.zeros_(self.output_head.weight); nn.init.zeros_(self.output_head.bias)
    def forward(self,emb,first_pass,site_ids=None,hours=None):
        B,T,_=emb.shape
        h=self.input_proj(torch.cat([emb,first_pass],dim=-1))+self.pos_enc[:,:T]
        if site_ids is not None and hours is not None:
            meta=self.meta_proj(torch.cat([self.site_emb(site_ids.clamp(0,self.site_emb.num_embeddings-1)),
                                            self.hour_emb(hours.clamp(0,23))],dim=-1))
            h=h+meta.unsqueeze(1)
        res=h; hf=self.ssm_fwd(h); hb=self.ssm_bwd(h.flip(1)).flip(1)
        h=self.ssm_norm(self.ssm_drop(self.ssm_merge(torch.cat([hf,hb],dim=-1)))+res)
        return self.output_head(h)

# ── Training helpers ──────────────────────────────────────────────────────────
def get_site_hour_ids(meta,file_list,site2i,n_cap=20):
    ftmp={}
    for f,s,h in zip(meta["filename"],meta["site"],meta["hour_utc"]):
        if f not in ftmp: ftmp[f]=(str(s),int(h))
    site_ids=np.zeros(len(file_list),dtype=np.int64); hour_ids=np.zeros(len(file_list),dtype=np.int64)
    for fi,fn in enumerate(file_list):
        if fn in ftmp:
            s,h=ftmp[fn]; site_ids[fi]=min(site2i.get(s,0),n_cap-1); hour_ids[fi]=h%24
    return site_ids,hour_ids

def train_light_proto_ssm(emb,sc_raw,Y,meta,n_epochs=40,patience=8,lr=1e-3,verbose=False):
    n_files=len(emb)//N_WINDOWS
    emb_f=emb.reshape(n_files,N_WINDOWS,-1); sc_f=sc_raw.reshape(n_files,N_WINDOWS,-1)
    Y_f=Y.reshape(n_files,N_WINDOWS,-1).astype(np.float32)
    sites=meta.drop_duplicates("filename")["site"].astype(str).tolist()
    site2i={s:i for i,s in enumerate(sorted(set(sites)))}
    site_ids,hour_ids=get_site_hour_ids(meta,meta.drop_duplicates("filename")["filename"].tolist(),site2i)
    model=LightProtoSSM(n_classes=N_CLASSES,n_sites=max(20,len(site2i)+1))
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-3)
    X=torch.tensor(emb_f,dtype=torch.float32); S=torch.tensor(sc_f,dtype=torch.float32)
    Yt=torch.tensor(Y_f,dtype=torch.float32)
    st=torch.tensor(site_ids,dtype=torch.long); ht=torch.tensor(hour_ids,dtype=torch.long)
    pos=Y_f.sum(axis=(0,1)); neg=Y_f.shape[0]*Y_f.shape[1]-pos
    pw=torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,25.0),dtype=torch.float32)
    best,wait,best_state=1e9,0,None
    for ep in range(n_epochs):
        model.train(); opt.zero_grad()
        out=model(X,S,st,ht)
        loss=F.binary_cross_entropy_with_logits(out,Yt,pos_weight=pw)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        v=float(loss.detach())
        if v<best: best,wait=v,0; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            wait+=1
            if wait>=patience: break
    if best_state: model.load_state_dict(best_state)
    model.eval(); return model,site2i

def run_tta_proto(model,emb_f,sc_f,site_t,hour_t,shifts=None,weights=None):
    if shifts is None: shifts=CFG["tta_shifts"]
    if weights is None: weights=CFG["tta_weights"]
    model.eval(); outs=[]
    emb_t=torch.tensor(emb_f,dtype=torch.float32); sc_t=torch.tensor(sc_f,dtype=torch.float32)
    for shift in shifts:
        e=torch.roll(emb_t,shift,dims=1) if shift else emb_t
        s=torch.roll(sc_t,shift,dims=1) if shift else sc_t
        with torch.no_grad(): out=model(e,s,site_ids=site_t,hours=hour_t).numpy()
        if shift: out=np.roll(out,-shift,axis=1)
        outs.append(out)
    return np.average(outs,axis=0,weights=weights)

def train_residual_ssm(emb_full,fp_flat,Y_full,site_ids,hour_ids,n_epochs=20,patience=6,lr=8e-4,cw=0.35):
    n_files=len(emb_full)//N_WINDOWS
    emb_f=emb_full.reshape(n_files,N_WINDOWS,-1); fp_f=fp_flat.reshape(n_files,N_WINDOWS,-1)
    lab_f=Y_full.reshape(n_files,N_WINDOWS,-1).astype(np.float32)
    residuals=lab_f-sigmoid(fp_f)
    model=ResidualSSM(n_classes=N_CLASSES)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-3)
    X=torch.tensor(emb_f,dtype=torch.float32); FP=torch.tensor(fp_f,dtype=torch.float32)
    R=torch.tensor(residuals,dtype=torch.float32)
    st=torch.tensor(site_ids,dtype=torch.long); ht=torch.tensor(hour_ids,dtype=torch.long)
    best,wait,best_state=1e9,0,None
    for ep in range(n_epochs):
        model.train(); opt.zero_grad()
        pred=model(X,FP,st,ht); loss=F.mse_loss(pred,R)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        v=float(loss.detach())
        if v<best: best,wait=v,0; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            wait+=1
            if wait>=patience: break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad(): correction=model(X,FP,st,ht).numpy()
    return model,(fp_f+cw*correction).reshape(-1,N_CLASSES).astype(np.float32)

# ── Test inference ────────────────────────────────────────────────────────────
banner('Test Inference', M)
test_paths=sorted((BASE/"test_soundscapes").glob("*.ogg"))
RUN_INFERENCE=len(test_paths)>0
if not RUN_INFERENCE:
    test_paths=sorted((BASE/"train_soundscapes").glob("*.ogg"))[:10]
    step("No test files — dry-run on 10 train soundscapes.")
else:
    info("Test files",str(len(test_paths)),G)
meta_te,sc_te,emb_te=run_perch(test_paths,CFG["batch_files"],verbose=True)

# ── Full pipeline ─────────────────────────────────────────────────────────────
banner('ProtoSSM Pipeline', M)
t0=time.time()

proto_model,site2i_tr=train_light_proto_ssm(emb_tr,sc_tr,Y_FULL_aligned,meta_tr,
                                              n_epochs=CFG["proto_epochs"],patience=CFG["proto_patience"])
n_te_f=len(sc_te)//N_WINDOWS; emb_te_f=emb_te.reshape(n_te_f,N_WINDOWS,-1); sc_te_f=sc_te.reshape(n_te_f,N_WINDOWS,-1)
n_tr_f=len(sc_tr)//N_WINDOWS; emb_tr_f=emb_tr.reshape(n_tr_f,N_WINDOWS,-1); sc_tr_f=sc_tr.reshape(n_tr_f,N_WINDOWS,-1)

test_fnames=meta_te.drop_duplicates("filename")["filename"].tolist()
tr_fnames  =meta_tr.drop_duplicates("filename")["filename"].tolist()
te_site_ids,te_hour_ids=get_site_hour_ids(meta_te,test_fnames,site2i_tr)
tr_site_ids,tr_hour_ids=get_site_hour_ids(meta_tr,tr_fnames,  site2i_tr)
te_site_t=torch.tensor(te_site_ids,dtype=torch.long); te_hour_t=torch.tensor(te_hour_ids,dtype=torch.long)
tr_site_t=torch.tensor(tr_site_ids,dtype=torch.long); tr_hour_t=torch.tensor(tr_hour_ids,dtype=torch.long)

# TTA on test
proto_te_flat=run_tta_proto(proto_model,emb_te_f,sc_te_f,te_site_t,te_hour_t).reshape(-1,N_CLASSES).astype(np.float32)
# TTA on train (for residual + calibration)
proto_tr_flat=run_tta_proto(proto_model,emb_tr_f,sc_tr_f,tr_site_t,tr_hour_t).reshape(-1,N_CLASSES).astype(np.float32)

prior_tables=build_prior_tables(sc,Y_SC)
sc_te_adj=apply_prior(sc_te,meta_te["site"].to_numpy(),meta_te["hour_utc"].to_numpy(),prior_tables)
sc_tr_adj=apply_prior(sc_tr,meta_tr["site"].to_numpy(),meta_tr["hour_utc"].to_numpy(),prior_tables)

probe_models,emb_scaler,emb_pca,alpha_blend=train_mlp_probes(emb_tr,sc_tr,Y_FULL_aligned)
sc_te_adj=apply_mlp_probes_vectorized(emb_te,sc_te_adj,probe_models,emb_scaler,emb_pca,alpha_blend)
sc_tr_adj=apply_mlp_probes_vectorized(emb_tr,sc_tr_adj,probe_models,emb_scaler,emb_pca,alpha_blend)

W=0.5
first_pass_te=(W*proto_te_flat+(1-W)*sc_te_adj).astype(np.float32)
first_pass_tr=(W*proto_tr_flat+(1-W)*sc_tr_adj).astype(np.float32)

PER_CLASS_THRESHOLDS=calibrate_and_optimize_thresholds(sigmoid(first_pass_tr),Y_FULL_aligned)
_,first_pass_tr_corr=train_residual_ssm(emb_tr,first_pass_tr,Y_FULL_aligned,tr_site_ids,tr_hour_ids,
                                          n_epochs=CFG["res_epochs"],patience=CFG["res_patience"])
res_model=ResidualSSM(n_classes=N_CLASSES)
# Reuse last trained res_model weights for test correction
with torch.no_grad():
    corr_te=res_model(
        torch.tensor(emb_te_f,dtype=torch.float32),
        torch.tensor(first_pass_te.reshape(n_te_f,N_WINDOWS,-1),dtype=torch.float32),
        site_ids=te_site_t,hours=te_hour_t).numpy().reshape(-1,N_CLASSES)
final_logits=first_pass_te+0.35*corr_te
final_logits=final_logits/CLASS_TEMPERATURES[None,:]

probs=sigmoid(final_logits)
probs=file_confidence_scale(probs); probs=np.clip(probs,0,1)
probs=rank_aware_scaling(probs); probs=np.clip(probs,0,1)
probs=adaptive_delta_smooth(probs); probs=np.clip(probs,0,1)
probs=apply_per_class_thresholds(probs,PER_CLASS_THRESHOLDS)
probs=np.clip(probs,0,1).astype(np.float32)

sub=pd.DataFrame(probs,columns=PRIMARY_LABELS)
sub.insert(0,"row_id",meta_te["row_id"].to_numpy())
sub.to_csv("submission_protossm.csv",index=False)
sub.to_csv("submission.csv",index=False)          # ← fallback in case SED cell fails
protossm_sub=sub.copy()

wall_time=time.time()-_WALL
banner('Cell 1 Done ✅', G)
info("submission_protossm.csv","saved",                             G)
info("Shape",                   str(sub.shape),                    G)
info("Wall time",               f"{wall_time:.1f}s ({wall_time/60:.1f} min)", Y)
print(f"\n{C}{sub.iloc[:3,:7].to_string()}{RS}")


═════════════════════════════════════════════════════════════════
  Cell 1 — ProtoSSM + ONNX Perch
═════════════════════════════════════════════════════════════════
  Classes                     234
  Full files                  59
  Active classes              71
  Perch backend               ONNX ✅
  Mapped species              203/234
  Proxy targets               3
▶  External cache: /kaggle/input/datasets/jaejohn/perch-meta
  sc_tr                       (708, 234)
  emb_tr                      (708, 1536)

═════════════════════════════════════════════════════════════════
  Test Inference
═════════════════════════════════════════════════════════════════
▶  No test files — dry-run on 10 train soundscapes.


Perch:   0%|          | 0/1 [00:00<?, ?it/s]


═════════════════════════════════════════════════════════════════
  ProtoSSM Pipeline
═════════════════════════════════════════════════════════════════
  Embedding→PCA               (708, 1536)→(708, 128) (89.96%)


MLP probes:   0%|          | 0/58 [00:00<?, ?it/s]

  MLP probes                  58/234

═════════════════════════════════════════════════════════════════
  Cell 1 Done ✅
═════════════════════════════════════════════════════════════════
  submission_protossm.csv     saved
  Shape                       (120, 235)
  Wall time                   57.3s (1.0 min)

                                     row_id   1161364    116570   1176823   1491113   1595929    209233
0   BC2026_Train_0001_S08_20250606_030007_5  0.028502  0.559540  0.014229  0.188364  0.001213  0.035076
1  BC2026_Train_0001_S08_20250606_030007_10  0.032405  0.549294  0.012486  0.186873  0.001043  0.036517
2  BC2026_Train_0001_S08_20250606_030007_15  0.031305  0.549931  0.013108  0.183370  0.001069  0.033209


<div style="background: linear-gradient(135deg, #0a1a0a 0%, #1a3a1a 50%, #0d2b0d 100%); padding: 35px 20px; border-radius: 12px; text-align: center; font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; box-shadow: 0 12px 24px rgba(0,0,0,0.6); border: 1px solid rgba(0, 230, 118, 0.15);">
    <p style="color: #ffffff; font-size: 22px; margin: 0 0 10px 0; font-weight: 700; letter-spacing: 0.5px;">🙌 Found this helpful?</p>
    <div style="height: 3px; width: 60px; background: linear-gradient(to right, #00e676, #1de9b6); margin: 0 auto 15px auto; border-radius: 2px; box-shadow: 0 0 10px rgba(0, 230, 118, 0.5);"></div>
    <p style="color: #a5d6a7; font-size: 15px; margin: 0 0 6px 0; letter-spacing: 0.5px;">If this notebook helped you, an upvote goes a long way — it helps others discover it too.</p>
    <p style="color: #6dada6; font-size: 14px; margin: 0; letter-spacing: 0.3px;">Feel free to fork, share, and build on top of it. Good luck in the competition! 🦜</p>
</div>